In [ ]:
## Fix up the environment for Windows Lab Machine
import os
# Remove invalid SSL override(s)
os.environ.pop("SSL_CERT_FILE", None)
os.environ.pop("SSL_CERT_DIR", None)
import warnings
warnings.filterwarnings("ignore")

# PyTorch is the primary deep learning framework we will use.
import torch

# The 'transformers' library by HuggingFace provides access to thousands of pre-trained models.
import transformers

# AutoTokenizer loads the vocabulary and text-to-number mapping for our specific model.
# AutoModelForSequenceClassification loads a model specifically designed with a classification head.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

# The datasets library makes it easy to download and process standard ML datasets.
from datasets import load_dataset, concatenate_datasets


In [ ]:
# We use DistilBERT, a smaller, faster, and lighter version of BERT.
model_name = "distilbert-base-uncased"

# Load the tokenizer that matches the model. It handles breaking text into tokens.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pre-trained model but replace the output layer with a new one for 2 classes (Positive/Negative).
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification


In [ ]:
# Load the IMDb movie review dataset. We are grabbing the 'train' split.
dataset = load_dataset("stanfordnlp/imdb", split="train")  # Load full train set
print("Loading..")


In [ ]:
# To train efficiently in a short lab, we downsample the dataset to 1,000 positive and 1,000 negative samples.
positive_samples = dataset.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(1000))
negative_samples = dataset.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(1000))

# Merge the positive and negative samples into a single balanced dataset.
balanced_dataset = concatenate_datasets([positive_samples, negative_samples]).shuffle(seed=42)

# This function converts raw text into numerical tensors that the model can understand.
def preprocess_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True,        # Cut off reviews longer than max_length
        padding="max_length",   # Pad shorter reviews with zeroes up to max_length
        max_length=256, 
        return_tensors="pt"     # Return PyTorch tensors ('pt')
    )

# Apply the preprocessing function to our dataset and remove the raw text column.
tokenized_dataset = balanced_dataset.map(preprocess_function, batched=True, remove_columns=["text"])


In [ ]:
print("Sample data after tokenization:")
# You can see 'input_ids' (the tokenized words) and 'attention_mask' (which tokens to ignore due to padding).
print(tokenized_dataset[:5])


In [ ]:
# TrainingArguments define the hyperparameters and saving strategies for the fine-tuning process.
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Evaluate the model's accuracy at the end of each epoch
    save_strategy="epoch",
    learning_rate=5e-5,          # How quickly the model updates its weights. 5e-5 is standard for fine-tuning.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,          # Number of full passes through the training dataset
    weight_decay=0.01,           # Regularization technique to prevent overfitting
    logging_dir="./logs",
    logging_steps=10,
    lr_scheduler_type="linear",  # Slowly decreases learning rate over time to stabilize training
)

print("Done")


In [ ]:
# Move the model to the GPU if one is available. GPUs drastically accelerate neural network training.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  

# The Trainer class handles the heavy lifting of the training loop.
trainer = Trainer(
    model=model,
    args=training_args,
    # Use 1,500 samples for training, and 500 for evaluating how well it's learning.
    train_dataset=tokenized_dataset.select(range(1500)),  
    eval_dataset=tokenized_dataset.select(range(1500, 2000)),  
)

# Begin the fine-tuning process!
trainer.train()


In [ ]:
# Once training is done, we save the fine-tuned weights and the tokenizer so we can use them later.
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")


In [ ]:
# This function wraps the entire inference pipeline for testing our model.
def classify_text(text):
    # Determine execution device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
    model.to(device)  

    # 1. Tokenize the input text exactly how we did during training
    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  

    # 2. Pass the tokens through the model to get the raw logits (un-normalized scores)
    outputs = model(**inputs)
    
    # 3. Find the index (0 or 1) of the highest score
    prediction = torch.argmax(outputs.logits).item()
    
    # 4. Map the index back to a human-readable label
    return "Positive" if prediction == 1 else "Negative"


In [ ]:
sample_text = "This movie was amazing! I loved every moment of it."
print("Sentiment:", classify_text(sample_text))


In [ ]:
# Let's test the model on a variety of unseen positive and negative sentences.

# Test Positive Sentences
print("Sentiment:", classify_text("I absolutely loved this film! It was fantastic."))
print("Sentiment:", classify_text("An incredible performance by the cast. A must-watch!"))
print("Sentiment:", classify_text("The storyline was beautiful and touching."))
print("Sentiment:", classify_text("This is one of the best movies I’ve ever seen."))
print("Sentiment:", classify_text("A masterpiece! Brilliant acting and a compelling plot."))

# Test Negative Sentences
print("Sentiment:", classify_text("This was the worst movie I have ever watched."))
print("Sentiment:", classify_text("Terrible acting, bad plot, and a complete waste of time."))
print("Sentiment:", classify_text("I was bored to death, nothing exciting ever happened."))
print("Sentiment:", classify_text("I regret watching this movie. It was awful."))
print("Sentiment:", classify_text("Disappointing in every way. Don't waste your time."))


In [ ]:
print("Sentiment:", classify_text("I absolutely loved this film! It was fantastic."))
print("Sentiment:", classify_text("An incredible performance by the cast. A must-watch!"))
print("Sentiment:", classify_text("The storyline was beautiful and touching."))
print("Sentiment:", classify_text("This is one of the best movies I’ve ever seen."))
print("Sentiment:", classify_text("A masterpiece! Brilliant acting and a compelling plot."))

# Test Negative Sentences
print("Sentiment:", classify_text("This was the worst movie I have ever watched."))
print("Sentiment:", classify_text("Terrible acting, bad plot, and a complete waste of time."))
print("Sentiment:", classify_text("I was bored to death, nothing exciting ever happened."))
print("Sentiment:", classify_text("I regret watching this movie. It was awful."))
print("Sentiment:", classify_text("Disappointing in every way. Don't waste your time."))
